# MCI-GRU Performance Proof Missing Grid Completion

This notebook completes the known missing cells from `performance_proof_tests/20260504_000039` without rerunning the already-completed 360 backtests.

It follows the repo notebook I/O rule: Google Drive is the durable source/sink, and `/content` is the active workspace. Market and regime inputs are staged locally before training, run artifacts are written locally, and compact outputs are synced back to Drive at the end.

It tests three areas:

- completes the missing year / variant / base-seed cells and their 15 scenario backtests;
- imports the prior 360-row decision table and compares `no_regime` against `full` in the completed grid;
- isolates the 2022 weakness by variant, seed, top-k, cost bucket, drawdown, turnover, and excess return.

## 1. Mount Drive, Clone Repo, Install Dependencies

Prepare the Colab VM, mount Drive, check out the intended branch, install dependencies, and fail early if a GPU is required but not visible.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/magilliam27/MCI-GRU.git'
BRANCH = 'main'
REPO_DIR = Path('/content/MCI-GRU') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/MCI-GRU-Ablations') if IN_COLAB else Path.cwd() / 'drive_outputs'
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/MCI_GRU_shared/data') if IN_COLAB else Path.cwd() / 'data' / 'raw' / 'market'
LOCAL_WORK_BASE = Path('/content/mci_gru_work') if IN_COLAB else Path.cwd() / 'local_work'
LOCAL_DATA_BASE = Path('/content/mci_gru_data') if IN_COLAB else Path.cwd() / 'local_data'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_WORK_BASE.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_BASE.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())
print('Drive output root:', DRIVE_ROOT)
print('Drive data folder:', DRIVE_DATA_DIR)
print('Local work base:', LOCAL_WORK_BASE)
print('Local data base:', LOCAL_DATA_BASE)

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev,tracking,fred]'], check=True)

REQUIRE_GPU = True
import torch
print('Python:', sys.executable)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
elif REQUIRE_GPU:
    raise RuntimeError('No CUDA GPU is visible. In Colab, switch Runtime -> Change runtime type -> GPU.')

## 2. FRED API Key

Use this only if `REGIME_INPUTS_CSV` is blank and a full-regime training job is enabled. Prefer a staged regime CSV when available.

In [ ]:
FRED_API_KEY = ''
if FRED_API_KEY:
    os.environ['FRED_API_KEY'] = FRED_API_KEY
print('FRED_API_KEY set:', bool(os.environ.get('FRED_API_KEY')))

## 3. Missing-Grid Configuration

All user-editable knobs live here. Defaults run full-budget confirmation for the three known missing training cells and their 45 scenario backtests.

In [ ]:
from datetime import datetime

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPERIMENT_SLUG = 'performance_proof_missing_grid'
LOCAL_WORK_ROOT = LOCAL_WORK_BASE / EXPERIMENT_SLUG / RUN_TAG
LOCAL_DATA_ROOT = LOCAL_DATA_BASE / EXPERIMENT_SLUG / RUN_TAG
DRIVE_EXPORT_ROOT = DRIVE_ROOT / EXPERIMENT_SLUG / RUN_TAG
RUN_ROOT = LOCAL_WORK_ROOT
TRAINING_OUTPUT_DIR = LOCAL_WORK_ROOT / 'training_runs'
PRIOR_STAGING_ROOT = LOCAL_WORK_ROOT / 'prior_results'
for path in [LOCAL_WORK_ROOT, LOCAL_DATA_ROOT, TRAINING_OUTPUT_DIR, PRIOR_STAGING_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

PRIOR_RUN_ROOT = DRIVE_ROOT / 'performance_proof_tests' / '20260504_000039'
PRIOR_DECISION_TABLE = PRIOR_RUN_ROOT / 'combined_proof_decision_table.csv'
PRIOR_POOLED_DAILY_RETURNS = PRIOR_RUN_ROOT / 'pooled_daily_returns.csv'
PRIOR_POOLED_SIGNIFICANCE = PRIOR_RUN_ROOT / 'pooled_daily_significance.csv'

RUN_TRAINING = True
RUN_BACKTESTS = True
IMPORT_PRIOR_RESULTS = True

NUM_MODELS = 20
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
BATCH_SIZE = 32
LEARNING_RATE = '5e-5'
BOOTSTRAP_RESAMPLES = 1000

DRIVE_DATA_ARCHIVE = ''
REGIME_INPUTS_CSV = ''  # Example: 'lseg_regime_export_20160101_20260504.csv'
REGIME_STRICT = True
REGIME_ENFORCE_LAG_DAYS = 0

MODEL_RECIPE = {
    'name': 'static-threshold-shuffle__pure-ic-returns-5d-val-ic__regime-current-only__ensemble__drop-edge-0p1',
    'label_t': 5,
    'loss_type': 'ic',
    'label_type': 'returns',
    'selection_metric': 'val_ic',
    'drop_edge_p': 0.1,
}

MISSING_TRAINING_CELLS = [
    {'test_year': 2022, 'variant': 'near_zero_graph', 'base_seed': 3141, 'data_config': 'temporal_2016', 'data_filename': 'sp500_2016_universe_data.csv', 'train_start': '2016-01-01', 'train_end': '2020-12-31', 'val_start': '2021-01-22', 'val_end': '2021-12-31', 'test_start': '2022-01-22', 'test_end': '2022-12-31'},
    {'test_year': 2023, 'variant': 'full', 'base_seed': 1729, 'data_config': 'temporal_2017', 'data_filename': 'sp500_2017_universe_data.csv', 'train_start': '2017-01-01', 'train_end': '2021-12-31', 'val_start': '2022-01-22', 'val_end': '2022-12-31', 'test_start': '2023-01-22', 'test_end': '2023-12-31'},
    {'test_year': 2023, 'variant': 'full', 'base_seed': 2718, 'data_config': 'temporal_2017', 'data_filename': 'sp500_2017_universe_data.csv', 'train_start': '2017-01-01', 'train_end': '2021-12-31', 'val_start': '2022-01-22', 'val_end': '2022-12-31', 'test_start': '2023-01-22', 'test_end': '2023-12-31'},
]

MODEL_VARIANTS = {
    'full': {'description': 'Momentum + regime + static threshold graph.', 'overrides': []},
    'near_zero_graph': {'description': 'Very high correlation threshold graph baseline.', 'overrides': ['graph.judge_value=0.9999', 'graph.top_k=0']},
    'no_regime': {'description': 'Global regime features removed.', 'overrides': ['features.include_global_regime=false', 'features.regime_strict=false', 'features.regime_include_subsequent_returns=false']},
}

EXPECTED_YEARS = [2022, 2023, 2024]
EXPECTED_VARIANTS = ['full', 'near_zero_graph', 'no_regime']
EXPECTED_BASE_SEEDS = [1729, 2718, 3141]
TOP_K_VALUES = [5, 10, 15, 20, 30]
COST_STRESS_GRID = [
    {'cost_name': 'spread5_slip0', 'spread_bps': 5.0, 'slippage_bps': 0.0},
    {'cost_name': 'spread10_slip2', 'spread_bps': 10.0, 'slippage_bps': 2.0},
    {'cost_name': 'spread20_slip5', 'spread_bps': 20.0, 'slippage_bps': 5.0},
]
RANK_DROP_GATE = {'enabled': True, 'min_rank_drop': 30}
HOLDING_PERIOD = 1
REBALANCE_STYLE = 'staggered'
NUM_TESTS_OVERRIDE = None

print('Local run root:', LOCAL_WORK_ROOT)
print('Drive export root:', DRIVE_EXPORT_ROOT)
print('Prior run root:', PRIOR_RUN_ROOT)
print('Missing training cells:', len(MISSING_TRAINING_CELLS))
print('Expected missing backtests:', len(MISSING_TRAINING_CELLS) * len(TOP_K_VALUES) * len(COST_STRESS_GRID))

## 4. Stage Drive Data Locally And Check Availability

Copy only the required market and optional regime inputs into local VM storage. Training and backtests use these local paths, not mounted Drive paths.

In [ ]:
import json
import pandas as pd

REQUIRED_DATA_FILENAMES = sorted({cell['data_filename'] for cell in MISSING_TRAINING_CELLS})
STAGED_DATA_FILES = {}
STAGED_REGIME_INPUTS_CSV = ''

if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f'Drive data folder not found: {DRIVE_DATA_DIR}')

def copy_once(src: Path, dst: Path) -> Path:
    if not src.exists():
        raise FileNotFoundError(f'Missing required source file: {src}')
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() or src.stat().st_size != dst.stat().st_size:
        shutil.copy2(src, dst)
    return dst

def stage_data_archive() -> Path | None:
    if not DRIVE_DATA_ARCHIVE:
        return None
    archive_src = DRIVE_DATA_DIR / DRIVE_DATA_ARCHIVE
    archive_dst = LOCAL_DATA_ROOT / archive_src.name
    copy_once(archive_src, archive_dst)
    extract_dir = LOCAL_DATA_ROOT / 'extracted'
    extract_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(archive_dst), str(extract_dir))
    return extract_dir

def find_staged_file(filename: str, extracted_dir: Path | None) -> Path | None:
    candidates = [
        LOCAL_DATA_ROOT / filename,
        LOCAL_DATA_ROOT / 'market' / filename,
        LOCAL_DATA_ROOT / 'data' / filename,
        LOCAL_DATA_ROOT / 'data' / 'raw' / 'market' / filename,
    ]
    if extracted_dir:
        candidates += [
            extracted_dir / filename,
            extracted_dir / 'market' / filename,
            extracted_dir / 'data' / filename,
            extracted_dir / 'data' / 'raw' / 'market' / filename,
        ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    if extracted_dir:
        matches = list(extracted_dir.rglob(filename))
        if matches:
            return matches[0].resolve()
    return None

extracted_data_dir = stage_data_archive()
if not DRIVE_DATA_ARCHIVE:
    print('DRIVE_DATA_ARCHIVE is blank; copying required CSV files once from Drive to local /content staging.')

for filename in REQUIRED_DATA_FILENAMES:
    staged = find_staged_file(filename, extracted_data_dir)
    if staged is None:
        staged = copy_once(DRIVE_DATA_DIR / filename, LOCAL_DATA_ROOT / 'market' / filename)
    STAGED_DATA_FILES[filename] = staged
    preview = pd.read_csv(staged, usecols=['dt', 'kdcode'])
    preview['dt'] = pd.to_datetime(preview['dt'])
    print(f"{filename}: {staged} | rows={len(preview):,}, stocks={preview.kdcode.nunique():,}, dates={preview.dt.min().date()} to {preview.dt.max().date()}")

if REGIME_INPUTS_CSV:
    regime_name = Path(REGIME_INPUTS_CSV).name
    regime_path = find_staged_file(regime_name, extracted_data_dir)
    if regime_path is None:
        repo_candidate = REPO_DIR / REGIME_INPUTS_CSV
        if repo_candidate.exists():
            regime_path = copy_once(repo_candidate, LOCAL_DATA_ROOT / 'regime' / regime_name)
        else:
            regime_path = copy_once(DRIVE_DATA_DIR / regime_name, LOCAL_DATA_ROOT / 'regime' / regime_name)
    STAGED_REGIME_INPUTS_CSV = regime_path.as_posix()
    print('Using staged regime CSV:', STAGED_REGIME_INPUTS_CSV)
elif REGIME_STRICT and any(cell['variant'] == 'full' for cell in MISSING_TRAINING_CELLS) and not os.environ.get('FRED_API_KEY'):
    raise RuntimeError('Full-regime missing cells are enabled. Set REGIME_INPUTS_CSV or FRED_API_KEY before training.')

## 5. Build Missing Training And Backtest Matrices

Construct deterministic Hydra overrides and make the intended 3 x 15 run shape explicit before expensive work starts.

In [ ]:
import itertools
import re

BASE_OVERRIDES = [
    'features=with_momentum',
    'data.source=csv',
    'tracking.enabled=true',
    'tracking.log_predictions=false',
    f'training.num_models={NUM_MODELS}',
    f'training.num_epochs={NUM_EPOCHS}',
    f'training.early_stopping_patience={EARLY_STOPPING_PATIENCE}',
    f'training.batch_size={BATCH_SIZE}',
    f'training.learning_rate={LEARNING_RATE}',
    f'evaluation.bootstrap_resamples={BOOTSTRAP_RESAMPLES}',
    'features.include_momentum=true',
    'features.include_weekly_momentum=true',
    'features.momentum_encoding=binary',
    'features.momentum_blend_mode=static',
    'features.momentum_blend_fast_weight=0.5',
    'features.include_global_regime=true',
    f'features.regime_strict={str(REGIME_STRICT).lower()}',
    f'features.regime_enforce_lag_days={REGIME_ENFORCE_LAG_DAYS}',
    'features.regime_include_subsequent_returns=false',
    'features.regime_change_months=12',
    'features.regime_norm_months=120',
    'features.regime_exclusion_months=1',
    'features.regime_similarity_quantile=0.2',
    'features.regime_min_history_months=24',
    'graph.judge_value=0.8',
    'graph.update_frequency_months=0',
    'graph.corr_lookback_days=252',
    'graph.top_k=0',
    'graph.top_k_metric=corr',
    'graph.use_multi_feature_edges=true',
    'graph.append_snapshot_age_days=false',
    'graph.use_lead_lag_features=false',
    'training.shuffle_train=true',
    f"training.loss_type={MODEL_RECIPE['loss_type']}",
    f"training.label_type={MODEL_RECIPE['label_type']}",
    f"training.selection_metric={MODEL_RECIPE['selection_metric']}",
    f"model.label_t={MODEL_RECIPE['label_t']}",
    f"graph.drop_edge_p={MODEL_RECIPE['drop_edge_p']}",
]
if STAGED_REGIME_INPUTS_CSV:
    BASE_OVERRIDES.append(f'features.regime_inputs_csv={STAGED_REGIME_INPUTS_CSV}')

def safe_name(value: str, max_len: int = 120) -> str:
    cleaned = re.sub(r'[^A-Za-z0-9_.-]+', '_', value).strip('_')
    return cleaned if len(cleaned) <= max_len else cleaned[:max_len]

def staged_data_path_for(cell: dict) -> str:
    staged_path = STAGED_DATA_FILES.get(cell['data_filename'])
    if staged_path is None or not Path(staged_path).exists():
        raise FileNotFoundError(f"Run local data staging first; missing {cell['data_filename']}")
    return Path(staged_path).resolve().as_posix()

training_jobs = []
for cell in MISSING_TRAINING_CELLS:
    variant_config = MODEL_VARIANTS[cell['variant']]
    name = safe_name(f"{MODEL_RECIPE['name']}__{cell['variant']}__base-seed-{cell['base_seed']}__test-{cell['test_year']}__missing-grid")
    overrides = [
        *BASE_OVERRIDES,
        *variant_config['overrides'],
        f"seed={cell['base_seed']}",
        f"data={cell['data_config']}",
        f"data.filename={staged_data_path_for(cell)}",
        f"data.train_start={cell['train_start']}",
        f"data.train_end={cell['train_end']}",
        f"data.val_start={cell['val_start']}",
        f"data.val_end={cell['val_end']}",
        f"data.test_start={cell['test_start']}",
        f"data.test_end={cell['test_end']}",
        f"output_dir={TRAINING_OUTPUT_DIR.as_posix()}",
        f"experiment_name={name}",
    ]
    training_jobs.append({**cell, 'staged_data_path': staged_data_path_for(cell), 'variant_description': variant_config['description'], 'name': name, 'overrides': overrides})

backtest_scenarios = []
for top_k, cost in itertools.product(TOP_K_VALUES, COST_STRESS_GRID):
    backtest_scenarios.append({
        'scenario': f"k{top_k}_{cost['cost_name']}_rankdrop{RANK_DROP_GATE['min_rank_drop']}_daily",
        'top_k': top_k,
        'transaction_costs': True,
        'spread_bps': cost['spread_bps'],
        'slippage_bps': cost['slippage_bps'],
        'rank_drop_gate': RANK_DROP_GATE['enabled'],
        'min_rank_drop': RANK_DROP_GATE['min_rank_drop'],
        'holding_period': HOLDING_PERIOD,
        'rebalance_style': REBALANCE_STYLE,
    })

training_matrix_df = pd.DataFrame(training_jobs)
scenario_df = pd.DataFrame(backtest_scenarios)
display(training_matrix_df[['test_year', 'variant', 'base_seed', 'name', 'data_filename', 'train_start', 'train_end', 'val_start', 'val_end', 'test_start', 'test_end']])
display(scenario_df)
print('Training jobs:', len(training_jobs))
print('Backtests after training:', len(training_jobs) * len(backtest_scenarios))

## 6. Run, Collect, And Score Helpers

Keep logs local, preserve failed rows, copy completed backtest folders into the run root, and flatten standard metrics into result rows.

In [ ]:
import time

def is_timestamp_dir(path: Path) -> bool:
    return path.is_dir() and bool(re.fullmatch(r'\d{8}_\d{6}', path.name))

def latest_run_dir(experiment_name: str) -> Path | None:
    base = TRAINING_OUTPUT_DIR / experiment_name
    if not base.exists():
        return None
    candidates = sorted([p for p in base.iterdir() if is_timestamp_dir(p)])
    return candidates[-1] if candidates else None

def flatten_dict(value: dict, prefix: str = '') -> dict:
    out = {}
    for key, item in value.items():
        full_key = f'{prefix}.{key}' if prefix else str(key)
        if isinstance(item, dict):
            out.update(flatten_dict(item, full_key))
        else:
            out[full_key] = item
    return out

def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def run_training_job(job: dict) -> dict:
    run_log_dir = RUN_ROOT / 'logs' / job['name']
    run_log_dir.mkdir(parents=True, exist_ok=True)
    stdout_path = run_log_dir / 'stdout.log'
    stderr_path = run_log_dir / 'stderr.log'
    cmd = [sys.executable, '-u', str(REPO_DIR / 'run_experiment.py'), *job['overrides']]
    print('\n' + '=' * 110)
    print('Training:', job['name'])
    print('Variant:', job['variant'], '| Base seed:', job['base_seed'], '| Test year:', job['test_year'])
    start = time.time()
    proc = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    elapsed = (time.time() - start) / 60
    stdout_path.write_text(proc.stdout, encoding='utf-8')
    stderr_path.write_text(proc.stderr, encoding='utf-8')
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print(proc.stderr[-4000:])
    run_dir = latest_run_dir(job['name'])
    row = {
        'status': 'OK' if proc.returncode == 0 else 'FAILED',
        'returncode': proc.returncode,
        'elapsed_minutes': elapsed,
        'run_dir': str(run_dir) if run_dir else '',
        'predictions_dir': str(run_dir / 'averaged_predictions') if run_dir else '',
        'staged_data_path': job.get('staged_data_path', ''),
        'stdout_log': str(stdout_path),
        'stderr_log': str(stderr_path),
        **{k: job[k] for k in ['test_year', 'variant', 'base_seed', 'name', 'data_config', 'data_filename', 'train_start', 'train_end', 'val_start', 'val_end', 'test_start', 'test_end']},
    }
    if run_dir:
        row.update({f'training_summary.{k}': v for k, v in flatten_dict(read_json(run_dir / 'training_summary.json')).items()})
        row.update({f'run_metadata.{k}': v for k, v in flatten_dict(read_json(run_dir / 'run_metadata.json')).items()})
    return row

def run_backtest(training_row: pd.Series, scenario: dict, num_tests: int) -> dict:
    pred_dir = Path(training_row['predictions_dir'])
    suffix = '_' + scenario['scenario']
    cmd = [
        sys.executable,
        str(REPO_DIR / 'tests' / 'backtest_sp500.py'),
        '--predictions_dir', str(pred_dir),
        '--data_file', str(training_row.get('staged_data_path') or STAGED_DATA_FILES[training_row['data_filename']]),
        '--test_start', training_row['test_start'],
        '--test_end', training_row['test_end'],
        '--top_k', str(scenario['top_k']),
        '--label_t', str(MODEL_RECIPE['label_t']),
        '--holding_period', str(scenario['holding_period']),
        '--rebalance_style', scenario['rebalance_style'],
        '--num_tests', str(num_tests),
        '--adjustment_method', 'bhy',
        '--auto_save',
        '--plot',
        '--disable_mlflow_autolink',
        '--backtest_suffix', suffix,
    ]
    if scenario['transaction_costs']:
        cmd.extend(['--transaction_costs', '--spread', str(scenario['spread_bps']), '--slippage', str(scenario['slippage_bps'])])
    if scenario['rank_drop_gate']:
        cmd.extend(['--enable_rank_drop_gate', '--min_rank_drop', str(scenario['min_rank_drop'])])
    print('\n' + '-' * 110)
    print('Backtest:', training_row['name'], '|', scenario['scenario'])
    proc = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print(proc.stderr[-3000:])
    source_dir = pred_dir.parent / f"backtest_{scenario['scenario']}"
    copy_dir = RUN_ROOT / 'backtests' / str(training_row['test_year']) / training_row['variant'] / f"base_seed_{training_row['base_seed']}" / scenario['scenario']
    if source_dir.exists():
        if copy_dir.exists():
            shutil.rmtree(copy_dir)
        shutil.copytree(source_dir, copy_dir)
    row = {
        'status': 'OK' if proc.returncode == 0 else 'FAILED',
        'returncode': proc.returncode,
        'test_year': training_row['test_year'],
        'variant': training_row['variant'],
        'base_seed': training_row['base_seed'],
        'name': training_row['name'],
        'scenario': scenario['scenario'],
        'predictions_dir': str(pred_dir),
        'source_backtest_dir': str(source_dir),
        'copied_backtest_dir': str(copy_dir),
        'stdout_tail': proc.stdout[-5000:],
        'stderr_tail': proc.stderr[-5000:],
        **{f'scenario_config.{k}': v for k, v in scenario.items()},
    }
    metrics = read_json(source_dir / 'backtest_metrics.json')
    row.update({f'backtest.{k}': v for k, v in metrics.items()})
    result_csv = source_dir / 'backtest_results.csv'
    if result_csv.exists():
        result_df = pd.read_csv(result_csv)
        if len(result_df):
            for key, value in result_df.iloc[0].to_dict().items():
                row.setdefault(f'backtest.{key}', value)
    return row

def add_decision_score(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    score = pd.Series(0.0, index=out.index)
    weights = {'backtest.ASR': 0.35, 'backtest.excess_return': 0.30, 'backtest.MDD': -0.20, 'backtest.avg_daily_turnover': -0.15}
    for col, weight in weights.items():
        if col in out.columns:
            vals = pd.to_numeric(out[col], errors='coerce')
            denom = vals.std(skipna=True)
            if pd.notna(denom) and denom != 0:
                score += weight * ((vals - vals.mean(skipna=True)) / denom).fillna(0.0)
    out['decision_score'] = score
    return out

## 7. Run Missing Training Cells And Backtests

Expensive cell. It writes interim CSVs after each training job and each backtest so a runtime interruption preserves progress.

In [ ]:
manifest_path = RUN_ROOT / 'performance_proof_missing_grid_manifest.json'
manifest_path.write_text(json.dumps({
    'run_tag': RUN_TAG,
    'local_work_root': str(LOCAL_WORK_ROOT),
    'local_data_root': str(LOCAL_DATA_ROOT),
    'drive_export_root': str(DRIVE_EXPORT_ROOT),
    'prior_run_root': str(PRIOR_RUN_ROOT),
    'drive_data_dir': str(DRIVE_DATA_DIR),
    'drive_data_archive': DRIVE_DATA_ARCHIVE,
    'staged_data_files': {k: str(v) for k, v in STAGED_DATA_FILES.items()},
    'model_recipe': MODEL_RECIPE,
    'missing_training_cells': MISSING_TRAINING_CELLS,
    'top_k_values': TOP_K_VALUES,
    'cost_stress_grid': COST_STRESS_GRID,
    'rank_drop_gate': RANK_DROP_GATE,
    'base_overrides': BASE_OVERRIDES,
    'budget': {'num_models': NUM_MODELS, 'num_epochs': NUM_EPOCHS, 'early_stopping_patience': EARLY_STOPPING_PATIENCE, 'bootstrap_resamples': BOOTSTRAP_RESAMPLES},
}, indent=2), encoding='utf-8')

training_rows = []
if RUN_TRAINING:
    for job in training_jobs:
        training_rows.append(run_training_job(job))
        pd.DataFrame(training_rows).to_csv(LOCAL_WORK_ROOT / 'missing_training_results_interim.csv', index=False)
else:
    for job in training_jobs:
        run_dir = latest_run_dir(job['name'])
        training_rows.append({'status': 'OK' if run_dir else 'FAILED', 'returncode': 0 if run_dir else 1, 'run_dir': str(run_dir) if run_dir else '', 'predictions_dir': str(run_dir / 'averaged_predictions') if run_dir else '', 'staged_data_path': job.get('staged_data_path', ''), **{k: job[k] for k in ['test_year', 'variant', 'base_seed', 'name', 'data_config', 'data_filename', 'train_start', 'train_end', 'val_start', 'val_end', 'test_start', 'test_end']}})

training_df = pd.DataFrame(training_rows)
training_results_path = RUN_ROOT / 'missing_training_results.csv'
training_df.to_csv(training_results_path, index=False)
display(training_df[[c for c in ['status', 'test_year', 'variant', 'base_seed', 'elapsed_minutes', 'run_dir', 'predictions_dir'] if c in training_df.columns]])

backtest_rows = []
if RUN_BACKTESTS:
    ok_training = training_df[training_df['status'].eq('OK')].copy()
    expected_full_tests = len(EXPECTED_YEARS) * len(EXPECTED_VARIANTS) * len(EXPECTED_BASE_SEEDS) * len(backtest_scenarios)
    num_tests = NUM_TESTS_OVERRIDE or expected_full_tests
    for _, train_row in ok_training.iterrows():
        for scenario in backtest_scenarios:
            backtest_rows.append(run_backtest(train_row, scenario, num_tests))
            pd.DataFrame(backtest_rows).to_csv(LOCAL_WORK_ROOT / 'missing_backtest_results_interim.csv', index=False)
else:
    num_tests = NUM_TESTS_OVERRIDE or 1

missing_backtest_df = pd.DataFrame(backtest_rows)
missing_raw_backtest_path = RUN_ROOT / 'missing_backtest_results_raw.csv'
missing_backtest_df.to_csv(missing_raw_backtest_path, index=False)
display(missing_backtest_df.head())
print('Manifest:', manifest_path)
print('Missing training results:', training_results_path)
print('Missing backtest raw results:', missing_raw_backtest_path)

## 8. Import Prior 360-Row Result Tables

Stage compact prior outputs locally. This imports the prior decision table and pooled daily returns once.

In [ ]:
prior_df = pd.DataFrame()
prior_pooled_daily_df = pd.DataFrame()
prior_pooled_sig_df = pd.DataFrame()

if IMPORT_PRIOR_RESULTS:
    if not PRIOR_DECISION_TABLE.exists():
        raise FileNotFoundError(f'Prior decision table not found: {PRIOR_DECISION_TABLE}')
    staged_prior_table = copy_once(PRIOR_DECISION_TABLE, PRIOR_STAGING_ROOT / PRIOR_DECISION_TABLE.name)
    prior_df = pd.read_csv(staged_prior_table)
    prior_df['source_table'] = str(PRIOR_DECISION_TABLE)
    prior_df['source_generation'] = 'prior_360'
    print('Imported prior decision rows:', len(prior_df), 'from', PRIOR_DECISION_TABLE)
    if PRIOR_POOLED_DAILY_RETURNS.exists():
        staged_prior_daily = copy_once(PRIOR_POOLED_DAILY_RETURNS, PRIOR_STAGING_ROOT / PRIOR_POOLED_DAILY_RETURNS.name)
        prior_pooled_daily_df = pd.read_csv(staged_prior_daily)
        prior_pooled_daily_df['source_generation'] = 'prior_360'
        print('Imported prior pooled daily rows:', len(prior_pooled_daily_df))
    if PRIOR_POOLED_SIGNIFICANCE.exists():
        staged_prior_sig = copy_once(PRIOR_POOLED_SIGNIFICANCE, PRIOR_STAGING_ROOT / PRIOR_POOLED_SIGNIFICANCE.name)
        prior_pooled_sig_df = pd.read_csv(staged_prior_sig)
        print('Imported prior pooled significance rows:', len(prior_pooled_sig_df))

prior_results_path = RUN_ROOT / 'imported_prior_360_results.csv'
prior_df.to_csv(prior_results_path, index=False)
display(prior_df.head())
print('Staged prior results:', prior_results_path)

## 9. Completed Grid Coverage And Decision Tables

Append the new 45 rows to the prior 360 rows, recompute the decision score on the completed table, and report any cells that remain missing.

In [ ]:
def normalize_result_types(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ['test_year', 'base_seed', 'scenario_config.top_k']:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    for col in out.columns:
        if col.startswith('backtest.') or col == 'decision_score':
            out[col] = pd.to_numeric(out[col], errors='ignore')
    return out

new_df = missing_backtest_df.copy()
if not new_df.empty:
    new_df['source_generation'] = 'missing_grid_completion'

completed_df = pd.concat([df for df in [prior_df, new_df] if not df.empty], ignore_index=True, sort=False)
if completed_df.empty:
    raise RuntimeError('No prior or new backtest rows are available.')
completed_df = add_decision_score(normalize_result_types(completed_df).drop(columns=['decision_score'], errors='ignore'))
if 'status' in completed_df.columns:
    completed_df['_status_rank'] = completed_df['status'].map({'OK': 0, 'FAILED': 1}).fillna(2)
else:
    completed_df['_status_rank'] = 0
completed_df = completed_df.sort_values(['_status_rank', 'decision_score'], ascending=[True, False]).drop(columns=['_status_rank'])

completed_raw_path = RUN_ROOT / 'completed_proof_results_raw.csv'
completed_decision_path = RUN_ROOT / 'completed_proof_decision_table.csv'
completed_html_path = RUN_ROOT / 'completed_proof_decision_table.html'
completed_df.to_csv(completed_raw_path, index=False)
completed_df.to_csv(completed_decision_path, index=False)
completed_df.to_html(completed_html_path, index=False)

expected_scenarios = sorted([s['scenario'] for s in backtest_scenarios])
expected_rows = [{'test_year': y, 'variant': v, 'base_seed': s, 'scenario': sc} for y, v, s, sc in itertools.product(EXPECTED_YEARS, EXPECTED_VARIANTS, EXPECTED_BASE_SEEDS, expected_scenarios)]
expected_df = pd.DataFrame(expected_rows)
if 'status' in completed_df.columns:
    completed_ok_df = completed_df[completed_df['status'].eq('OK')]
else:
    completed_ok_df = completed_df
actual_ok_keys = completed_ok_df[['test_year', 'variant', 'base_seed', 'scenario']].drop_duplicates()
coverage_df = expected_df.merge(actual_ok_keys, on=['test_year', 'variant', 'base_seed', 'scenario'], how='left', indicator=True)
coverage_df['coverage_status'] = coverage_df['_merge'].map({'both': 'present', 'left_only': 'missing'})
coverage_df = coverage_df.drop(columns=['_merge'])
coverage_path = RUN_ROOT / 'completed_grid_coverage.csv'
coverage_df.to_csv(coverage_path, index=False)
missing_after_df = coverage_df[coverage_df['coverage_status'].eq('missing')]
missing_after_path = RUN_ROOT / 'remaining_missing_grid_rows.csv'
missing_after_df.to_csv(missing_after_path, index=False)
cell_coverage = coverage_df.assign(present=coverage_df['coverage_status'].eq('present').astype(int)).groupby(['test_year', 'variant', 'base_seed'], dropna=False)['present'].agg(['sum', 'count']).reset_index().rename(columns={'sum': 'present_scenarios', 'count': 'expected_scenarios'})
cell_coverage['missing_scenarios'] = cell_coverage['expected_scenarios'] - cell_coverage['present_scenarios']
cell_coverage_path = RUN_ROOT / 'completed_grid_cell_coverage.csv'
cell_coverage.to_csv(cell_coverage_path, index=False)

display_cols = [c for c in ['status', 'source_generation', 'test_year', 'variant', 'base_seed', 'scenario', 'decision_score', 'backtest.ARR', 'backtest.ASR', 'backtest.MDD', 'backtest.total_return_calendar_aligned', 'backtest.benchmark_return', 'backtest.excess_return', 'backtest.avg_daily_turnover', 'backtest.haircutted_sharpe', 'backtest.adjusted_p_value'] if c in completed_df.columns]
display(completed_df[display_cols].head(40))
display(cell_coverage)
print('Completed rows:', len(completed_df))
print('Expected full grid rows:', len(expected_df))
print('Remaining missing rows:', len(missing_after_df))

## 10. Main Effects, Full vs No-Regime, And 2022 Diagnostics

Core interpretation layer for the three questions: coverage, `no_regime` vs `full`, and 2022 weakness.

In [ ]:
summary_dir = RUN_ROOT / 'summaries'
summary_dir.mkdir(exist_ok=True)
ok = completed_df[completed_df['status'].eq('OK')].copy() if 'status' in completed_df.columns else completed_df.copy()
if 'scenario_config.top_k' in ok.columns:
    ok['top_k'] = pd.to_numeric(ok['scenario_config.top_k'], errors='coerce')
else:
    ok['top_k'] = ok['scenario'].str.extract(r'k(\d+)_')[0].astype(float)
ok['cost_bucket'] = ok['scenario'].str.extract(r'(spread\d+_slip\d+)')

metric_cols = [c for c in ['decision_score', 'backtest.ARR', 'backtest.ASR', 'backtest.MDD', 'backtest.total_return_calendar_aligned', 'backtest.benchmark_return', 'backtest.excess_return', 'backtest.IR', 'backtest.avg_daily_turnover', 'backtest.total_transaction_cost', 'backtest.cost_drag_ARR', 'backtest.haircutted_sharpe'] if c in ok.columns]

for group_cols in [['test_year'], ['variant'], ['base_seed'], ['top_k'], ['cost_bucket'], ['scenario'], ['test_year', 'variant'], ['variant', 'top_k'], ['variant', 'cost_bucket']]:
    table = ok.groupby(group_cols, dropna=False)[metric_cols].agg(['mean', 'median', 'count'])
    name = 'summary_by_' + '_'.join(group_cols)
    table.to_csv(summary_dir / f'{name}.csv')
    display(table)

pair_keys = ['test_year', 'base_seed', 'scenario']
full_no_regime_compare = pd.DataFrame()
compare_base = ok[ok['variant'].isin(['full', 'no_regime'])].copy()
pivots = []
for metric in ['backtest.excess_return', 'backtest.ARR', 'backtest.ASR', 'backtest.MDD', 'decision_score']:
    if metric in compare_base.columns:
        pivot = compare_base.pivot_table(index=pair_keys, columns='variant', values=metric, aggfunc='mean')
        pivot.columns = [f'{metric}.{col}' for col in pivot.columns]
        pivots.append(pivot)
if pivots:
    full_no_regime_compare = pd.concat(pivots, axis=1).reset_index()
    for metric in ['backtest.excess_return', 'backtest.ARR', 'backtest.ASR', 'decision_score']:
        full_col = f'{metric}.full'
        no_regime_col = f'{metric}.no_regime'
        if full_col in full_no_regime_compare.columns and no_regime_col in full_no_regime_compare.columns:
            full_no_regime_compare[f'{metric}.no_regime_minus_full'] = full_no_regime_compare[no_regime_col] - full_no_regime_compare[full_col]
    compare_path = summary_dir / 'full_vs_no_regime_pairwise.csv'
    full_no_regime_compare.to_csv(compare_path, index=False)
    display(full_no_regime_compare.sort_values('backtest.excess_return.no_regime_minus_full', ascending=False).head(30))
    print('Full vs no_regime pairwise:', compare_path)

weak_2022 = ok[ok['test_year'].eq(2022)].copy() if 'test_year' in ok.columns else pd.DataFrame()
if not weak_2022.empty:
    for group_cols in [['variant'], ['base_seed'], ['top_k'], ['cost_bucket'], ['variant', 'top_k'], ['variant', 'cost_bucket']]:
        table = weak_2022.groupby(group_cols, dropna=False)[metric_cols].agg(['mean', 'median', 'count'])
        table.to_csv(summary_dir / ('diagnostic_2022_by_' + '_'.join(group_cols) + '.csv'))
        display(table)
    worst_2022 = weak_2022.sort_values(['backtest.excess_return', 'backtest.MDD'], ascending=[True, True])
    worst_2022_path = summary_dir / 'diagnostic_2022_worst_rows.csv'
    worst_2022.to_csv(worst_2022_path, index=False)
    display(worst_2022[[c for c in ['variant', 'base_seed', 'scenario', 'decision_score', 'backtest.ARR', 'backtest.ASR', 'backtest.MDD', 'backtest.excess_return', 'backtest.avg_daily_turnover', 'backtest.total_transaction_cost'] if c in worst_2022.columns]].head(25))

## 11. Completed-Grid Pooled Daily Excess-Return Significance

Combine prior pooled daily rows with the new missing-grid daily returns, then recompute pooled significance on the completed grid.

In [ ]:
import numpy as np
from scipy import stats

def daily_returns_path(row: pd.Series) -> Path | None:
    for key in ['source_backtest_dir', 'copied_backtest_dir']:
        if key in row and pd.notna(row[key]):
            path = Path(str(row[key])) / 'daily_returns.csv'
            if path.exists():
                return path
    return None

new_pooled_rows = []
new_ok = new_df[new_df['status'].eq('OK')].copy() if not new_df.empty else pd.DataFrame()
for _, row in new_ok.iterrows():
    path = daily_returns_path(row)
    if path is None:
        continue
    daily = pd.read_csv(path)
    if 'portfolio_return' not in daily.columns or 'benchmark_return' not in daily.columns:
        continue
    daily['date'] = pd.to_datetime(daily['date'])
    daily['excess_return_daily'] = daily['portfolio_return'].astype(float) - daily['benchmark_return'].astype(float)
    for _, drow in daily.iterrows():
        new_pooled_rows.append({'test_year': row.get('test_year'), 'variant': row.get('variant'), 'base_seed': row.get('base_seed'), 'scenario': row.get('scenario'), 'date': drow['date'], 'portfolio_return': drow['portfolio_return'], 'benchmark_return': drow['benchmark_return'], 'excess_return_daily': drow['excess_return_daily'], 'source_generation': 'missing_grid_completion'})

new_pooled_daily_df = pd.DataFrame(new_pooled_rows)
new_pooled_daily_path = RUN_ROOT / 'missing_grid_pooled_daily_returns.csv'
new_pooled_daily_df.to_csv(new_pooled_daily_path, index=False)
completed_pooled_daily_df = pd.concat([df for df in [prior_pooled_daily_df, new_pooled_daily_df] if not df.empty], ignore_index=True, sort=False) if (not prior_pooled_daily_df.empty or not new_pooled_daily_df.empty) else pd.DataFrame()
completed_pooled_daily_path = RUN_ROOT / 'completed_pooled_daily_returns.csv'
completed_pooled_daily_df.to_csv(completed_pooled_daily_path, index=False)

sig_rows = []
if not completed_pooled_daily_df.empty:
    completed_pooled_daily_df['date'] = pd.to_datetime(completed_pooled_daily_df['date'])
    for keys, group in completed_pooled_daily_df.groupby(['variant', 'scenario'], dropna=False):
        vals = group['excess_return_daily'].dropna().astype(float)
        if len(vals) < 3:
            continue
        mean_daily = vals.mean()
        t_stat, p_value = stats.ttest_1samp(vals, 0.0)
        ann_excess = (1 + mean_daily) ** 252 - 1
        ann_vol = vals.std(ddof=1) * np.sqrt(252)
        sig_rows.append({'variant': keys[0], 'scenario': keys[1], 'n_days': len(vals), 'mean_daily_excess': mean_daily, 'annualized_excess': ann_excess, 'annualized_excess_vol': ann_vol, 'annualized_information_ratio': ann_excess / ann_vol if ann_vol else np.nan, 't_statistic': t_stat, 'p_value_two_sided': p_value})
completed_pooled_sig_df = pd.DataFrame(sig_rows).sort_values(['variant', 'annualized_information_ratio'], ascending=[True, False]) if sig_rows else pd.DataFrame()
completed_pooled_sig_path = RUN_ROOT / 'completed_pooled_daily_significance.csv'
completed_pooled_sig_df.to_csv(completed_pooled_sig_path, index=False)
display(completed_pooled_sig_df)
print('New pooled daily rows:', len(new_pooled_daily_df))
print('Completed pooled daily rows:', len(completed_pooled_daily_df))

## 12. Visualizations

Compact diagnostics for scanability. Use the CSVs for exact values.

In [ ]:
import matplotlib.pyplot as plt

plot_dir = RUN_ROOT / 'plots'
plot_dir.mkdir(exist_ok=True)
plot_metrics = [(c, t) for c, t in [('backtest.ASR', 'Annualized Sharpe'), ('backtest.excess_return', 'Excess Return'), ('backtest.MDD', 'Maximum Drawdown'), ('backtest.avg_daily_turnover', 'Average Daily Turnover')] if c in ok.columns]
if plot_metrics:
    compact = ok.copy()
    compact['label'] = compact.get('test_year', '').astype(str) + ' | ' + compact.get('variant', '').astype(str) + ' | ' + compact.get('base_seed', '').astype(str) + ' | ' + compact.get('scenario', '').astype(str)
    compact = compact.sort_values('decision_score', ascending=False).head(80)
    fig, axes = plt.subplots(len(plot_metrics), 1, figsize=(15, max(4, 4 * len(plot_metrics))))
    if len(plot_metrics) == 1:
        axes = [axes]
    for ax, (col, title) in zip(axes, plot_metrics):
        plot_df = compact.sort_values(col, ascending=True)
        ax.barh(plot_df['label'], pd.to_numeric(plot_df[col], errors='coerce'), color='#2f6f73')
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title(title)
        ax.grid(True, axis='x', alpha=0.25)
    plt.tight_layout()
    metric_plot_path = plot_dir / 'completed_grid_metric_bars_top80.png'
    plt.savefig(metric_plot_path, dpi=160, bbox_inches='tight')
    plt.show()

if not weak_2022.empty and {'variant', 'top_k', 'backtest.excess_return'}.issubset(weak_2022.columns):
    pivot = weak_2022.pivot_table(index='top_k', columns='variant', values='backtest.excess_return', aggfunc='mean')
    ax = pivot.plot(kind='bar', figsize=(11, 6), width=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title('2022 Mean Excess Return By Top-K And Variant')
    ax.set_ylabel('Mean excess return')
    ax.grid(True, axis='y', alpha=0.25)
    plt.tight_layout()
    diagnostic_2022_plot_path = plot_dir / 'diagnostic_2022_excess_by_topk_variant.png'
    plt.savefig(diagnostic_2022_plot_path, dpi=160, bbox_inches='tight')
    plt.show()

primary_scenario = f"k10_spread5_slip0_rankdrop{RANK_DROP_GATE['min_rank_drop']}_daily"
if not completed_pooled_daily_df.empty:
    fig, ax = plt.subplots(1, 1, figsize=(14, 7))
    for keys, group in completed_pooled_daily_df.groupby(['variant', 'scenario'], dropna=False):
        if keys[1] != primary_scenario:
            continue
        group = group.sort_values('date')
        values = np.cumprod(1 + group['excess_return_daily'].astype(float))
        ax.plot(group['date'], values, label=f'{keys[0]} | {keys[1]}', linewidth=1.4)
    ax.axhline(1.0, color='gray', linestyle=':', linewidth=1.0)
    ax.set_title('Completed-Grid Pooled Excess Return Curves - Primary Top-10 Low-Cost Scenario')
    ax.set_ylabel('Cumulative excess value')
    ax.grid(True, alpha=0.25)
    ax.legend(loc='best', fontsize=8)
    plt.tight_layout()
    pooled_excess_plot_path = plot_dir / 'completed_pooled_excess_curves_primary.png'
    plt.savefig(pooled_excess_plot_path, dpi=160, bbox_inches='tight')
    plt.show()

## 13. Failed-Run Inspection

Failures stay in the raw tables. This cell prints log tails so failures are visible experiment findings, not hidden notebook noise.

In [ ]:
def print_tail(path: str, n_chars: int = 4000):
    p = Path(path)
    if p.exists():
        text = p.read_text(encoding='utf-8', errors='replace')
        print(f'\n--- {p} ---')
        print(text[-n_chars:])

failed_training = training_df[~training_df['status'].eq('OK')] if not training_df.empty else pd.DataFrame()
if failed_training.empty:
    print('No failed training runs.')
else:
    display(failed_training)
    for _, row in failed_training.iterrows():
        print_tail(row.get('stdout_log', ''))
        print_tail(row.get('stderr_log', ''))

failed_backtests = missing_backtest_df[~missing_backtest_df['status'].eq('OK')] if not missing_backtest_df.empty else pd.DataFrame()
if failed_backtests.empty:
    print('No failed missing-grid backtests.')
else:
    display(failed_backtests[[c for c in ['test_year', 'variant', 'base_seed', 'scenario', 'stdout_tail', 'stderr_tail'] if c in failed_backtests.columns]])

if not missing_after_df.empty:
    print('Rows still missing from the completed grid:')
    display(missing_after_df)
else:
    print('The expected 405-row grid is complete.')

## 14. Summary Report Export

Write the Markdown summary locally. The final sync cell copies it and compact artifacts back to Drive.

In [ ]:
def fmt_pct(value):
    if pd.isna(value):
        return ''
    return f'{100 * float(value):.2f}%'

report_table = completed_df[[c for c in ['status', 'source_generation', 'test_year', 'variant', 'base_seed', 'scenario', 'decision_score', 'backtest.ARR', 'backtest.ASR', 'backtest.MDD', 'backtest.total_return_calendar_aligned', 'backtest.benchmark_return', 'backtest.excess_return', 'backtest.avg_daily_turnover', 'backtest.haircutted_sharpe', 'backtest.adjusted_p_value'] if c in completed_df.columns]].copy()
for pct_col in ['backtest.ARR', 'backtest.MDD', 'backtest.total_return_calendar_aligned', 'backtest.benchmark_return', 'backtest.excess_return', 'backtest.avg_daily_turnover']:
    if pct_col in report_table.columns:
        report_table[pct_col] = report_table[pct_col].apply(fmt_pct)

new_table = new_df[[c for c in ['status', 'test_year', 'variant', 'base_seed', 'scenario', 'decision_score', 'backtest.ARR', 'backtest.ASR', 'backtest.MDD', 'backtest.excess_return', 'backtest.avg_daily_turnover'] if c in new_df.columns]].copy() if not new_df.empty else pd.DataFrame()
for pct_col in ['backtest.ARR', 'backtest.MDD', 'backtest.excess_return', 'backtest.avg_daily_turnover']:
    if pct_col in new_table.columns:
        new_table[pct_col] = new_table[pct_col].apply(fmt_pct)

report_path = RUN_ROOT / 'performance_proof_missing_grid_summary.md'
lines = [
    '# MCI-GRU Missing Grid Completion Summary',
    '',
    f'Local run root: `{LOCAL_WORK_ROOT}`',
    f'Local data root: `{LOCAL_DATA_ROOT}`',
    f'Drive export root: `{DRIVE_EXPORT_ROOT}`',
    f'Prior run root: `{PRIOR_RUN_ROOT}`',
    f'Model recipe: `{MODEL_RECIPE["name"]}`',
    '',
    '## Coverage',
    '',
    f'- Completed decision rows: `{len(completed_df)}`',
    f'- Expected full-grid rows: `{len(expected_df)}`',
    f'- Remaining missing rows: `{len(missing_after_df)}`',
    '',
    cell_coverage.to_markdown(index=False),
    '',
    '## Newly Produced Missing-Grid Rows',
    '',
    new_table.to_markdown(index=False) if not new_table.empty else 'No missing-grid backtest rows were produced.',
    '',
    '## Completed Decision Table',
    '',
    report_table.head(120).to_markdown(index=False),
    '',
    '## Completed Pooled Significance',
    '',
    completed_pooled_sig_df.to_markdown(index=False) if not completed_pooled_sig_df.empty else 'No pooled daily-return rows were available.',
    '',
    '## Artifacts',
    '',
    f'- Manifest: `{manifest_path}`',
    f'- Missing training results: `{training_results_path}`',
    f'- Missing raw backtest results: `{missing_raw_backtest_path}`',
    f'- Prior imported results: `{prior_results_path}`',
    f'- Completed raw results: `{completed_raw_path}`',
    f'- Completed decision table: `{completed_decision_path}`',
    f'- Completed HTML decision table: `{completed_html_path}`',
    f'- Coverage report: `{coverage_path}`',
    f'- Remaining missing rows: `{missing_after_path}`',
    f'- Completed pooled daily returns: `{completed_pooled_daily_path}`',
    f'- Completed pooled significance: `{completed_pooled_sig_path}`',
]
for maybe_path_name in ['metric_plot_path', 'diagnostic_2022_plot_path', 'pooled_excess_plot_path']:
    if maybe_path_name in globals():
        lines.append(f'- Plot: `{globals()[maybe_path_name]}`')
report_path.write_text('\n'.join(lines), encoding='utf-8')
print(report_path.read_text(encoding='utf-8')[:8000])
print('Summary report:', report_path)

## 15. Package Local Results And Sync Compact Artifacts To Drive

Export compact artifacts and a zipped local archive. This avoids streaming detailed intermediate files to Drive during the expensive run.

In [ ]:
archive_path = Path(shutil.make_archive(str(LOCAL_WORK_ROOT), 'zip', root_dir=str(LOCAL_WORK_ROOT.parent), base_dir=LOCAL_WORK_ROOT.name))
print('Local archive:', archive_path)

DRIVE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_PATHS = [
    manifest_path,
    training_results_path,
    missing_raw_backtest_path,
    prior_results_path,
    completed_raw_path,
    completed_decision_path,
    completed_html_path,
    coverage_path,
    cell_coverage_path,
    missing_after_path,
    new_pooled_daily_path,
    completed_pooled_daily_path,
    completed_pooled_sig_path,
    report_path,
    archive_path,
]
for maybe_path_name in ['metric_plot_path', 'diagnostic_2022_plot_path', 'pooled_excess_plot_path']:
    if maybe_path_name in globals():
        EXPORT_PATHS.append(globals()[maybe_path_name])

exported = []
for src in EXPORT_PATHS:
    src = Path(src)
    if not src.exists():
        continue
    dst = DRIVE_EXPORT_ROOT / src.name
    shutil.copy2(src, dst)
    exported.append(dst)

print('Synced compact artifacts to Drive export root:', DRIVE_EXPORT_ROOT)
for dst in exported:
    print('-', dst)